In [7]:
#step 1: import libraries


import pandas as pd
import numpy as np

import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report , confusion_matrix


In [8]:
# STEP 2: CREATE A SMALL BEGINNER DATASET
# ---------------------------------------------------------

# We are creating the dataset manually so it can run directly in Colab
# label = spam or ham
# message = the text message

data = {
    "label": [
        "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam",
        "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam"
    ],
    "message": [
        "Hey, are we meeting today?",
        "Congratulations! You won a free iPhone. Click now!",
        "Please send me the notes of today class",
        "Win cash prize now by clicking this link",
        "Can you call me after lunch?",
        "Limited offer! Buy now and get 50 percent discount",
        "I am on the way to college",
        "You have been selected for a free vacation",
        "Let's study together in the evening",
        "Claim your lottery reward today",
        "Can you help me with Python homework?",
        "Urgent! Your account has won a huge reward",
        "Where are you now?",
        "Free entry in contest, click here now",
        "I will reach home by 8 PM",
        "Get a free gift voucher instantly",
        "Did you complete the assignment?",
        "Exclusive deal only for you, act fast",
        "Let's go for tea after class",
        "You are a lucky winner, claim your prize"
    ]
}

# Convert dictionary into DataFrame
df = pd.DataFrame(data)

# Show first 5 rows
print("First 5 rows of dataset:")
display(df.head())

First 5 rows of dataset:


,label,message
0,ham,"Hey, are we meeting today?"
1,spam,Congratulations! You won a free iPhone. Click ...
2,ham,Please send me the notes of today class
3,spam,Win cash prize now by clicking this link
4,ham,Can you call me after lunch?


In [9]:
#srep 3 : check dataset information
print("\nDataset information:")
print(df.shape)
print("\ncolumn names:")
print(df.columns)
print("\ndataset info:")
df.info()
print("\nMissing values:")
print(df.isnull().sum())



Dataset information:
(20, 2)

column names:
Index(['label', 'message'], dtype='object')

dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    20 non-null     object
 1   message  20 non-null     object
dtypes: object(2)
memory usage: 452.0+ bytes

Missing values:
label      0
message    0
dtype: int64


In [10]:
# step 4 :Text Cleaning Function
def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove no
    text = re.sub(r'\d+' , '', text)

    #remove special char and punctuation
    text = re.sub(r'[^a-zA-Z\s]' , '', text)

    #remove extra space
    text = re.sub(r'\s+' , ' ' , text).strip()

    return text

df["clean_message"] = df["message"].apply(clean_text)
print("\nDataset after text cleaning:")
print(df[["message" , "clean_message"]].head())





Dataset after text cleaning:
                                             message  \
0                         Hey, are we meeting today?   
1  Congratulations! You won a free iPhone. Click ...   
2            Please send me the notes of today class   
3           Win cash prize now by clicking this link   
4                       Can you call me after lunch?   

                                     clean_message  
0                         hey are we meeting today  
1  congratulations you won a free iphone click now  
2          please send me the notes of today class  
3         win cash prize now by clicking this link  
4                      can you call me after lunch  


In [11]:
#step 5 : label encoding
label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["label"])
print("\nOriginal and encoded labels:")
print(df[["label" , "label_encoded"]].head())
print("\nlabel mapping:")
for i , class_name in enumerate(label_encoder.classes_):
    print(f"{class_name} --> {i}")


Original and encoded labels:
  label  label_encoded
0   ham              0
1  spam              1
2   ham              0
3  spam              1
4   ham              0

label mapping:
ham --> 0
spam --> 1


In [12]:
#step 6 : separate features and targets
x = df["clean_message"]
y = df["label_encoded"]
print("\nFeature samples:")
print(x.head())
print("\nTarget samples:")
print(y.head())


Feature samples:
0                           hey are we meeting today
1    congratulations you won a free iphone click now
2            please send me the notes of today class
3           win cash prize now by clicking this link
4                        can you call me after lunch
Name: clean_message, dtype: object

Target samples:
0    0
1    1
2    0
3    1
4    0
Name: label_encoded, dtype: int64


In [13]:
#step 7 : train_test_split
x_train , x_test , y_train , y_test = train_test_split(
    x, y, test_size=0.2 , random_state=42
)
print("\nTrain_test Split Shape:")
print("x_train:" , x_train.shape)
print("x_test:" , x_test.shape)
print("y_train:" , y_train.shape)
print("y_test:" , y_test.shape)


Train_test Split Shape:
x_train: (16,)
x_test: (4,)
y_train: (16,)
y_test: (4,)


In [14]:
#step 8 : convert text into no
vectorizer = CountVectorizer()
x_train_vectorized = vectorizer.fit_transform(x_train)
x_test_vectorized = vectorizer.transform(x_test)
print("\nshape after vectorization:")
print("x_train_vectorized:" , x_train_vectorized.shape)
print("x_test_vectorized:" , x_test_vectorized.shape)
print("\nsample vocabulary words:")
print(list(vectorizer.vocabulary_.keys())[:10])


shape after vectorization:
x_train_vectorized: (16, 74)
x_test_vectorized: (4, 74)

sample vocabulary words:
['lets', 'study', 'together', 'in', 'the', 'evening', 'limited', 'offer', 'buy', 'now']


In [15]:
# step 9 : train the model
model = MultinomialNB()
model.fit(x_train_vectorized , y_train)
print("\nModel Trained Sucessfully!")


Model Trained Sucessfully!


In [16]:
# step 10 : Make Prediction
y_pred = model.predict(x_test_vectorized)
print("\nPrediction Values:")
print(y_pred)
print("\nActual Values:")
print(y_test.values)


Prediction Values:
[0 0 1 1]

Actual Values:
[0 1 1 1]


In [17]:
#step 11 : evaluate the model
accuracy = accuracy_score(y_test , y_pred)
print("\nAccuracy:" , accuracy)
print("\nClassification Report:")
print(classification_report(y_test , y_pred))
print("\n Confusion Matrix:")
print(confusion_matrix(y_test , y_pred))


Accuracy: 0.75

Classification Report:
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       1.00      0.67      0.80         3

    accuracy                           0.75         4
   macro avg       0.75      0.83      0.73         4
weighted avg       0.88      0.75      0.77         4


 Confusion Matrix:
[[1 0]
 [1 2]]


In [22]:


#step 12 : Test with New Custom Messages
new_messages = [
    "Congratulations You Have Won a Gift",
    "Can You Send Me a Class Link",
    "Claim Your Cash Prize Now",
    "I am Coming in 10 Minutes",
    "Exclusive Offer Only Toady Buy Now"
]
cleaned_new_messages = [clean_text(msg) for msg in new_messages]
new_messages_vectorized = vectorizer.transform(cleaned_new_messages)
new_predictions = model.predict(new_messages_vectorized)
print("\nNew Message Predictions:")
for msg , prediction in zip(new_messages , new_predictions):
  label_name = label_encoder.inverse_transform([prediction])[0]
  print(f"Message:{msg}")
  print(f"Prediction:{label_name}\n")
  print("-"*50)



New Message Predictions:
Message:Congratulations You Have Won a Gift
Prediction:spam

--------------------------------------------------
Message:Can You Send Me a Class Link
Prediction:ham

--------------------------------------------------
Message:Claim Your Cash Prize Now
Prediction:spam

--------------------------------------------------
Message:I am Coming in 10 Minutes
Prediction:ham

--------------------------------------------------
Message:Exclusive Offer Only Toady Buy Now
Prediction:spam

--------------------------------------------------
